Retrieve research topics from OpenAlex for all works that cite the original pymatgen paper.

Then use LLM to organize topics into a mindmap.

Note:
- You would need to set `OPENAI_API_KEY` to use OpenAI API, see https://openai.com/api/.

In [ ]:
# Step 1: Get topics from OpenAlex for 1st pymatgen paper

from collections import Counter

import requests

params = {"filter": "cites:w2015197254", "group_by": "primary_topic.id", "per_page": 50}
response = requests.get(
    "https://api.openalex.org/works", params=params, timeout=10
).json()

topic_counter = Counter(
    {
        group.get("key_display_name", "None"): group["count"]
        for group in response.get("group_by")
        if group["count"] >= 10  # Count for top 50 papers is around 10
    }
)

print("Topics from OpenAlex:")
for topic, count in topic_counter.items():
    print(f"{topic:<60}{count}")

Topics from OpenAlex:
Machine Learning in Materials Science                       1144
Advanced Battery Materials and Technologies                 323
Advancements in Battery Materials                           322
Perovskite Materials and Applications                       107
Electrocatalysts for Energy Conversion                      75
Metal-Organic Frameworks: Synthesis and Applications        74
Electronic and Structural Properties of Oxides              67
2D Materials and Applications                               66
Advanced Thermoelectric Materials and Devices               55
MXene and MAX Phase Materials                               49
Synthesis and Properties of Inorganic Cluster Compounds     41
X-ray Diffraction in Crystallography                        37
Chalcogenide Semiconductor Thin Films                       35
Advancements in Solid Oxide Fuel Cells                      31
High Entropy Alloys Studies                                 31
Ammonia Synthesis and Nitrog

In [ ]:
# Step 2: Use LLM to group topics

import re

from openai import OpenAI

client = OpenAI()  # https://platform.openai.com/docs/overview

topic_list = "\n".join(f"- {topic} ({count})" for topic, count in topic_counter.items())

prompt = f"""
The following research topics are from papers that cite pymatgen, with each number indicating the count of citing works in that area.

Please organize them into 3-6 thematic branches that represent broader areas of materials research (e.g., batteries, machine learning, catalysis, ...).

Return output in a plain text tree structure (with indentation) without any header or metadata, e.g.:
    Topic_0:
        subtopic_0 (count_0)
        subtopic_1 (count_1)

    Topic_1:
        subtopic_0 (count_0)

Topics to work on today:
    {topic_list}
"""

chat_response = client.responses.create(
    model="gpt-4.1",
    input=prompt,
)

# print(chat_response.output_text)


def parse_mindmap(text: str) -> dict[str, list[tuple[str, int]]]:
    """
    Parse LLM response to machine-readable dict for plotter.
    """
    mindmap: dict[str, list[tuple[str, int]]] = {}
    current_topic = None

    for line in text.strip().splitlines():
        line = line.rstrip()

        if not line.strip():
            continue

        if not line.startswith(" "):  # Top-level topic
            current_topic = line.strip()
            mindmap[current_topic] = []

        elif current_topic:
            subtopic_line = line.strip()

            # Extract the (name, count)
            if match := re.match(r"^(.*)\s+\((\d+)\)$", subtopic_line):
                name = match[1].strip()
                count = int(match[2])
                mindmap[current_topic].append((name, count))
            else:
                raise ValueError(f"Cannot extract info from {subtopic_line=}")

    return mindmap


mindmap_dict: dict[str, list[tuple[str, int]]] = parse_mindmap(
    chat_response.output_text
)

for main_topic, subtopics in mindmap_dict.items():
    print(f"{main_topic}")
    for name, count in subtopics:
        print(f"    - {name} ({count})")

Energy Storage and Conversion:
    - Advanced Battery Materials and Technologies (323)
    - Advancements in Battery Materials (322)
    - Advanced battery technologies research (17)
    - Advancements in Solid Oxide Fuel Cells (31)
    - Hydrogen Storage and Materials (18)
    - Thermal Expansion and Ionic Conductivity (16)
    - Chemical Looping and Thermochemical Processes (13)
Catalysis and Sustainable Chemistry:
    - Electrocatalysts for Energy Conversion (75)
    - Catalytic Processes in Materials Science (25)
    - Advanced Photocatalysis Techniques (21)
    - Ammonia Synthesis and Nitrogen Reduction (27)
    - CO2 Reduction Techniques and Catalysts (19)
    - Zeolite Catalysis and Synthesis (10)
    - Catalysis and Oxidation Reactions (13)
Machine Learning, Computing, and Data Science:
    - Machine Learning in Materials Science (1144)
    - Scientific Computing and Data Management (13)
    - Computational Drug Discovery Methods (14)
Functional and Emerging Materials:
    - Pe

In [ ]:
# Step 3: Plot mindmap with Latex
# Ref: https://github.com/janosh/diagrams/blob/main/assets/physics-mindmap/physics-mindmap.tex

import os
import subprocess

mindmap_filename: str = "pymatgen_mindmap.tex"


def write_tikz_mindmap(
    mindmap: dict[str, list[tuple[str, int]]],
    filename: str,
    central_node: str = "pymatgen",
    max_subtopics_per_branch: int = 3,
) -> None:
    tikz_colors: list[str] = [
        "Gold",
        "DarkBlue",
        "Teal",
        "Red",
        "Purple",
        "DarkGreen",
        "RoyalBlue",
        "Orchid",
        "Brown",
        "CornflowerBlue",
        "SlateGray",
    ]

    with open(filename, "w", encoding="utf-8") as f:
        f.write(r"\documentclass[tikz, svgnames]{standalone}" + "\n")
        f.write(r"\usetikzlibrary{mindmap}" + "\n")
        f.write(r"\begin{document}" + "\n")
        f.write(r"\begin{tikzpicture}[" + "\n")
        f.write(
            r"    mindmap, every node/.style=concept, concept color=orange, text=white,"
            + "\n"
        )
        f.write(
            r"    level 1/.append style={level distance=5cm, sibling angle=60, font=\large},"
            + "\n"
        )
        f.write(
            r"    level 2/.append style={level distance=3cm, sibling angle=45}" + "\n"
        )
        f.write(r"  ]" + "\n\n")

        f.write(
            rf"  \node{{\textbf{{\huge{{{central_node}}}}}}} [clockwise from=0]" + "\n"
        )

        for i, (main_topic, subtopics) in enumerate(mindmap.items()):
            color = tikz_colors[i % len(tikz_colors)]
            direction = "clockwise" if i % 2 == 0 else "counterclockwise"
            from_angle = 60 + i * 30

            # Limit number of subtopics
            limited_subtopics = sorted(subtopics, key=lambda x: x[1], reverse=True)[
                :max_subtopics_per_branch
            ]

            f.write(f"  child [concept color={color}, text=black] {{\n")
            f.write(f"      node {{{main_topic}}} [{direction} from={from_angle}]\n")
            for name, _count in limited_subtopics:
                safe_name = (
                    name.replace("&", r"\&").replace("-", r"\-").replace("_", r"\_")
                )
                f.write(f"      child {{ node {{{safe_name}}} }}\n")
            f.write("    }\n")

        f.write(";\n")
        f.write(r"\end{tikzpicture}" + "\n")
        f.write(r"\end{document}" + "\n")


write_tikz_mindmap(mindmap_dict, mindmap_filename)

subprocess.run(
    ["pdflatex", "-interaction=nonstopmode", mindmap_filename],
    check=True,
    stdout=subprocess.DEVNULL,
)

for ext in (".aux", ".log"):
    try:
        os.remove(os.path.splitext(mindmap_filename)[0] + ext)
    except FileNotFoundError:
        pass